## Setup

In [6]:
import requests
import base64
from IPython.display import Image, display
from mlserver.codecs import NumpyRequestCodec, PandasCodec, NumpyCodec
from mlserver.types import InferenceResponse
import pandas as pd

## Core 1

### SeldonDeployment

#### Deploy

https://docs.seldon.ai/seldon-core-1

In [ ]:
# kubectl apply -f - << EOF
# apiVersion: machinelearning.seldon.io/v1
# kind: SeldonDeployment
# metadata:
#   name: iris-model
#   namespace: seldon-core-1
# spec:
#   name: iris
#   predictors:
#   - graph:
#       implementation: SKLEARN_SERVER
#       modelUri: gs://seldon-models/v1.19.0/sklearn/iris
#       name: classifier
#     name: default
#     replicas: 1
# EOF

In [8]:
%%bash

kubectl apply -f - << EOF
apiVersion: machinelearning.seldon.io/v1
kind: SeldonDeployment
metadata:
  name: iris-model
  namespace: seldon-core-1
spec:
  name: iris
  predictors:
  - name: default
    replicas: 1
    svcOrchSpec:                  # seldon-container-engine resources
      resources:
        requests:
          cpu: 50m
          memory: 64Mi
        limits:
          cpu: 500m
          memory: 512Mi
    graph:
      implementation: SKLEARN_SERVER
      modelUri: gs://seldon-models/v1.19.0/sklearn/iris
      name: classifier
    componentSpecs:
    - spec:
        containers:
        - name: classifier        # my server container
          resources:
            requests:
              cpu: 50m
              memory: 64Mi
            limits:
              cpu: 500m
              memory: 512Mi
EOF

seldondeployment.machinelearning.seldon.io/iris-model created


#### Inference

```bash
kubectl port-forward -n seldon-core-1 svc/iris-model-default 8080:8000  
```

In [7]:
requests.post(
    "http://localhost:8080/api/v1.0/predictions",
    json={"data": {"ndarray": [[1, 2, 3, 4]]}}
).json()

{'data': {'names': ['t:0', 't:1', 't:2'],
  'ndarray': [[0.0006985194531162835,
    0.00366803903943666,
    0.995633441507447]]},
 'meta': {'requestPath': {'classifier': 'seldonio/sklearnserver:1.19.0'}}}

or

In [13]:
CORE1_IP = !kubectl get svc istio-ingressgateway -n istio-system -o jsonpath='{.status.loadBalancer.ingress[0].ip}'
CORE1_IP = CORE1_IP[0]
print(CORE1_IP)

10.51.48.200


In [ ]:
requests.post(
    f"http://{CORE1_IP}/seldon/seldon-core-1/iris-model/api/v1.0/predictions",
    json={"data": {"ndarray": [[1, 2, 3, 4]]}}
).json()

{'data': {'names': ['t:0', 't:1', 't:2'],
  'ndarray': [[0.0006985194531162835,
    0.00366803903943666,
    0.995633441507447]]},
 'meta': {'requestPath': {'classifier': 'seldonio/sklearnserver:1.19.0'}}}

## Core 2

**Routing**

1. Tunnel directly to seldon-mesh (without LoadBalancer)
    Forward svc/istio-ingressgateway resource from the local machine’s port 8080 to port 80 of the istio-ingressgateway on the Kubernetes cluster:

    ```bash
        kubectl port-forward -n istio-system svc/istio-ingressgateway 8080:80 
    ```

2. With LoadBalancer:

In [16]:
CORE2_IP = !kubectl get svc seldon-mesh -n seldon-mesh -o jsonpath='{{.status.loadBalancer.ingress[0].ip}}'  
CORE2_IP = CORE2_IP[0]
print(CORE2_IP)

10.51.48.201


### Model

#### Deploy

In [1]:
%%bash
kubectl apply -f - --namespace=seldon-mesh <<EOF
apiVersion: mlops.seldon.io/v1alpha1
kind: Model
metadata:
  name: iris
spec:
  storageUri: "gs://seldon-models/scv2/samples/mlserver_1.3.5/iris-sklearn"
  requirements:
    - sklearn
EOF

model.mlops.seldon.io/iris created


#### Inference

In [ ]:
requests.post(
    f"http://{CORE2_IP}/v2/models/iris/infer",
    headers={
        "Seldon-Model": "iris",
        "Host": f"seldon-mesh.inference.seldon",
    },
    json={
        "inputs": [
            {
                "name": "predict",
                "shape": [1, 4],
                "datatype": "FP32",
                "data": [[1, 2, 3, 4]],
            }
        ]
    },
).json()

{'model_name': 'iris_1',
 'model_version': '1',
 'id': 'e12b84fb-4754-4f41-bf18-8256ca125523',
 'parameters': {},
 'outputs': [{'name': 'predict',
   'shape': [1, 1],
   'datatype': 'INT64',
   'parameters': {'content_type': 'np'},
   'data': [2]}]}

### Pipeline

In [13]:
def generate_mermaid_graph(graph):
    base64_string = base64.b64encode(graph.encode("utf8")).decode("ascii")
    display(Image(url="https://mermaid.ink/img/" + base64_string))

In [19]:
generate_mermaid_graph(
    """
    graph LR
        X[X] --> |encoding| I[Input]
        I[Input] --> PP
        subgraph Pipeline
            PP([Preprocessor]) --> M([Model])
        end
        M --> O[Output]
    """
)

#### Deploy

In [20]:
%%bash
kubectl apply -f - --namespace=seldon-mesh <<EOF
apiVersion: mlops.seldon.io/v1alpha1
kind: Model
metadata:
    name: preprocessor
    namespace: seldon-mesh
spec:
    storageUri: "gs://maciej-seldon/tutorials/pipelines/single-model/models/preprocessor"
    artifactVersion: 1        
    requirements:
      - mlserver
    memory: 200Ki
---
apiVersion: mlops.seldon.io/v1alpha1
kind: Model
metadata:
    name: xgboost-fraud-detection
    namespace: seldon-mesh
spec:
    storageUri: "gs://maciej-seldon/tutorials/pipelines/single-model/models/xgboost-fraud-detection"
    artifactVersion: 1        
    requirements:
      - xgboost
    memory: 200Ki
EOF

model.mlops.seldon.io/preprocessor created
model.mlops.seldon.io/xgboost-fraud-detection created


In [22]:
%%bash
kubectl apply -f - --namespace=seldon-mesh <<EOF
apiVersion: mlops.seldon.io/v1alpha1
kind: Pipeline
metadata:
  name: chained-models
  namespace: seldon-mesh
spec:
  steps:
    - name: preprocessor
    - name: xgboost-fraud-detection
      inputs:
      - preprocessor
  output:
    steps:  
    - xgboost-fraud-detection
EOF

pipeline.mlops.seldon.io/chained-models created


#### Inference

In [41]:
df = pd.DataFrame([{
    "Customer_Age": 36,
    "Policy_Duration": 39,
    "Vehicle_Age": 12,
    "Vehicle_Value": 12865,
    "Annual_Mileage": 7754,
    "No_Claims_Bonus": 7,
}])

request = PandasCodec.encode_request(
    df,
    use_bytes=False,
)
request.model_dump()

{'parameters': {'content_type': 'pd'},
 'inputs': [{'name': 'Customer_Age',
   'shape': [1, 1],
   'datatype': 'INT64',
   'data': [36]},
  {'name': 'Policy_Duration',
   'shape': [1, 1],
   'datatype': 'INT64',
   'data': [39]},
  {'name': 'Vehicle_Age', 'shape': [1, 1], 'datatype': 'INT64', 'data': [12]},
  {'name': 'Vehicle_Value',
   'shape': [1, 1],
   'datatype': 'INT64',
   'data': [12865]},
  {'name': 'Annual_Mileage',
   'shape': [1, 1],
   'datatype': 'INT64',
   'data': [7754]},
  {'name': 'No_Claims_Bonus',
   'shape': [1, 1],
   'datatype': 'INT64',
   'data': [7]}]}

In [ ]:
# requests.post(
#     f"http://{CORE2_IP}/v2/models/preprocessor/infer",
#     json=PandasCodec.encode_request(
#         df[df["Is_Fraudulent"] == 1].iloc[[0]].drop(columns=["Is_Fraudulent"]),
#         use_bytes=False,
#     ).model_dump(),
#     headers={
#         "Seldon-Model": "preprocessor",
#         "Host": "seldon=mesh.inference.seldon",
#     },
# ).json()["outputs"][0]["data"]

In [ ]:
# # transform input for genuine data
# X_inference_genuine_transformed = numeric_transformer_loaded.transform(
#     df[df["Is_Fraudulent"] == 0]
#     .reset_index(drop=True)
#     .loc[[0], numeric_features]
# )

# # construct inference request object
# model_request_genuine = NumpyRequestCodec.encode_request(
#     X_inference_genuine_transformed
# )

# # make inference request
# model_response_genuine = requests.post(
#     f"http://{CORE2_IP}/v2/models/{MODEL_NAME}/infer",
#     json=model_request_genuine.model_dump(),
#     headers={
#         "Seldon-Model": f"{MODEL_NAME}",
#         "Host": f"{NAMESPACE}.inference.seldon",
#     },
# )

# # decode inference response
# print(
#     NumpyRequestCodec.decode_response(
#         InferenceResponse.model_validate_json(model_response_genuine.text)
#     )
# )  # or model_response.json()["outputs"][0]["data"]

In [ ]:
requests.post(
    f'http://{CORE2_IP}/v2/models/chained-models/infer',
    json=request.model_dump(),
    headers={'Seldon-Model': 'chained-models.pipeline', 'Host': 'seldon.inference.seldon'},
).json()["outputs"][0]["data"][0]

1

## Experiment